In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Question 1.2.3:  On simule deux files indépendantes, avec des arrivées à taux constant et des départs à taux constant qui vaut 1.2 pour les arrivées et 1.5 pour les départs. On regarde le premier temps d'atteinte de 0 peu importe la file, et on garde en mémoire les différentes valeurs qu'ont pris les files bid et ask au cours des évenements, le temps d'occurence des différents évènements, et le temps d'atteinte de 0 pour une des files. 

In [ ]:
def deux_files(init, lam_plus, lam_minus):
    t = 0
    A = init
    V = init
    # Tableaux pour stocker les données de temps, ventes et achats
    temps = [t]
    ventes = [A]
    achats = [V]

    # Simulation jusqu'à ce que l'une des files atteigne 0
    while A > 0 and V > 0:
        # Calcul du temps jusqu'au prochain événement
        t += np.random.exponential(1 / (2 * (lam_plus + lam_minus)))
        
        # Détermination de l'événement (vente ou achat)
        if np.random.rand() < 0.5:

            # Si c'est une vente, on décide si c'est une augmentation ou diminution de la file de vente
            if np.random.rand() < lam_plus / (lam_plus + lam_minus):
                A += 1
            else:
                A -= 1
        else:
            # Si c'est un achat, on décide si c'est une augmentation ou diminution de la file d'achat
            if np.random.rand() < lam_plus / (lam_plus + lam_minus):
                V += 1
            else:
                V -= 1

        # Stockage des données après chaque événement
        temps.append(t)
        ventes.append(A)
        achats.append(V)

    return t, temps, ventes, achats

On fait de même mais en ne regardant qu'une seule file cette fois ci, et en gardant en mémoire le temps l'occurence de chaque incrémentation, ou diminution, et l'évolution de la file à chaque evenement, et finalemenet le temps d'atteinte de 0. 

In [ ]:
def une_file(init, lam_plus, lam_minus):
    t = 0.0
    q = init
    temps = [t]
    valeurs = [q]

    while q > 0:
        t += np.random.exponential(1 / (lam_plus + lam_minus))
        if np.random.rand() < lam_plus / (lam_plus + lam_minus):
            q += 1
        else:
            q -= 1
        temps.append(t)
        valeurs.append(q)

    return temps, valeurs, t

   

1.2.3.3: On veut simuler le temps moyen d’atteinte de zéro d’une des file d’attente et calculer un intervalle
de confiance.
On simule alors N trajectoires et one calcule la moyenne et de l'intervalle de confiance à 95% pour le temps d'atteinte de 0

In [ ]:
def moyenne_temps_atteinte(N, init, lam_plus, lam_minus):

    tt = []
    # Boucle pour simuler N fois et stocker les temps d'atteinte de 0
    for _ in range(N):
        # On simule une file et garder le temps d'atteinte de 0
        _, _, tau = une_file(init, lam_plus, lam_minus)
        tt.append(tau)
    tt = np.array(tt)

    # Calcul de la moyenne, de l'écart-type et de l'intervalle de confiance à 95%
    m = np.mean(tt)
    s = np.std(tt, ddof=1)
    ic = 1.96 * s / np.sqrt(N)
    # On retourne la moyenne et l'intervalle de confiance
    return m, m - ic, m + ic

On affiche de la valeur moyenne et de l'intervalle de confiance sur N = 1000 essais.

In [ ]:
N = 1e3
m, m_moins, m_plus = moyenne_temps_atteinte(N, init=10, lam_plus=1.2, lam_minus=1.5)
print(f"Le temps moyen d'atteinte de 0 est: {m:.2f}. L'intervalle de confiance à 95% est [{m_moins:.2f}, {m_plus:.2f}])")

1.2.3.5:  On simule la probabilité que 𝑄1 (0) touche zéro avant 𝑄−1(0) en fonction de 𝑄1 (0) et 𝑄−1(0).
On simule N fois les deux files et on compte combien de fois la file de vente atteint 0 avant la file d'achat

In [ ]:
def prob_Q1(N, init, lam_plus, lam_moins):
    inc = 0
    for _ in range(N):
        _, _, ventes, achats = deux_files(init, lam_plus, lam_moins)

        # On trouve l'index où l'une des files atteint 0
        for i in range(1, len(ventes)):
            if ventes[i] == 0:
                inc += 1
                break
            elif achats[i] == 0:
                break
    
    proba = inc / N
    return proba

Question 1.2.4: Processus de Hawkes 
1.2.4.1: Simuler une file d’attente de Hawkes et approcher son intensité stationnaire 𝜆−


In [ ]:
def hawkes_une_file(init, mu_plus, mu_moins, alpha, beta):
    # Initialisation des variables
    t = 0
    q = init
    z_plus = 0
    z_moins = 0
    lam_plus = mu_plus
    temps = [t]
    tailles = [q]
    intensites_moins = []


    # Simulation jusqu'à ce que la file atteigne 0
    while q > 0:
        # Calcul des intensités à l'instant t
        lam_minus = mu_moins - alpha * z_plus + alpha * z_moins
        # Intensité totale
        total = lam_plus + lam_minus

        # temps jusqu'au prochain événement
        dt = np.random.exponential(1 / total)

        # Mise à jour des intensités en fonction du temps écoulé
        z_plus *= np.exp(-beta * dt)
        z_moins *= np.exp(-beta * dt)

        # Incrémentation du temps
        t += dt
   
    # Détermination de l'événement (augmentation ou diminution de la file)
        if np.random.random() < lam_plus / total:
            q += 1
            z_plus += 1
        else:
            q -= 1
            z_moins += 1
    # Stockage des données après chaque événement
        temps.append(t)
        tailles.append(q)
        intensites_moins.append(lam_minus)

    return t, temps, tailles, intensites_moins

1.2.4.2: Estimer la distribution des temps d’atteinte de zero par cette file d’attente.